In [1]:
# Imports and Setup
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

DATA_PATH = r"C:\Users\LA-Mahdis\Desktop\Maciek\ENEN-645\Final_Project\Dataset"

cuda:0


In [2]:
from pathlib import Path

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

USE_KAGGLE = False

if USE_KAGGLE:
    import kagglehub
    import shutil

    KAGGLE_HANDLE = "maciekpopik/plantlab2realgeneralization"

    dataset_path = kagglehub.dataset_download(KAGGLE_HANDLE)
    dataset_path = Path(dataset_path)

    if is_colab():
        TARGET_DIR = Path("/content/plant_dataset")

        # Clean existing folder
        if TARGET_DIR.exists():
            shutil.rmtree(TARGET_DIR)

        # Copy from cache → /content
        shutil.copytree(dataset_path, TARGET_DIR)

        OUTDIR = str(TARGET_DIR)
    else:
        TARGET_DIR = Path(DATA_PATH)

        if not TARGET_DIR.exists():
            shutil.copytree(dataset_path, TARGET_DIR)

    OUTDIR = str(TARGET_DIR)

    print("Dataset ready at:", OUTDIR)

else:
    # Local Root dataset folder
    OUTDIR = DATA_PATH
    print("Using local dataset at:", OUTDIR)

Using local dataset at: C:\Users\LA-Mahdis\Desktop\Maciek\ENEN-645\Final_Project\Dataset


In [3]:
# === Directory setup ===

train_dir = os.path.join(OUTDIR, "Train")
val_dir   = os.path.join(OUTDIR, "Val")
test_id_dir  = os.path.join(OUTDIR, "Test_ID")
test_ood_dir = os.path.join(OUTDIR, "Test_OOD")

print("Train:", train_dir)
print("Val:", val_dir)
print("Test ID:", test_id_dir)
print("Test OOD:", test_ood_dir)

Train: C:\Users\LA-Mahdis\Desktop\Maciek\ENEN-645\Final_Project\Dataset\Train
Val: C:\Users\LA-Mahdis\Desktop\Maciek\ENEN-645\Final_Project\Dataset\Val
Test ID: C:\Users\LA-Mahdis\Desktop\Maciek\ENEN-645\Final_Project\Dataset\Test_ID
Test OOD: C:\Users\LA-Mahdis\Desktop\Maciek\ENEN-645\Final_Project\Dataset\Test_OOD


In [4]:
# =====================================================================
# Define image size for resizing images and compute training data stats
# =====================================================================

IMG_SIZE = 224

def compute_mean_std(image_folder, batch_size=64, num_workers=0):
    from tqdm import tqdm

    temp_dataset = datasets.ImageFolder(
        image_folder,
        transform=transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])
    )

    temp_loader = DataLoader(
        temp_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    n_pixels = 0
    channel_sum = torch.zeros(3)
    channel_sum_sq = torch.zeros(3)

    total_images = len(temp_dataset)
    print(f"Processing {total_images} images...")
    for images, _ in tqdm(temp_loader):
        # images shape: [B, C, H, W]
        b, c, h, w = images.shape
        pixels_per_batch = b * h * w

        channel_sum += images.sum(dim=[0, 2, 3])
        channel_sum_sq += (images ** 2).sum(dim=[0, 2, 3])
        n_pixels += pixels_per_batch

    print()
    mean = channel_sum / n_pixels
    std = torch.sqrt(channel_sum_sq / n_pixels - mean ** 2)

    return mean, std

train_mean, train_std = compute_mean_std(train_dir, batch_size=64, num_workers=0)
print("Mean:", train_mean)
print("Std :", train_std)

Processing 27694 images...


100%|██████████| 433/433 [01:21<00:00,  5.29it/s]


Mean: tensor([0.4689, 0.4986, 0.4264])
Std : tensor([0.1997, 0.1730, 0.2131])


In [5]:
# === Defining transforms used on data ===

transform = {
    "train": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=train_mean.tolist(), std=train_std.tolist())
    ]),
    "val": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=train_mean.tolist(), std=train_std.tolist())
    ]),
    "test": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=train_mean.tolist(), std=train_std.tolist())
    ]),
}

In [6]:
# === Custom ImageFolder that also returns file path ===

class ImageFolderWithPaths(datasets.ImageFolder):
    def __getitem__(self, index):
        image, label = super().__getitem__(index)
        path, _ = self.samples[index]
        return image, label, path

In [7]:
# === Creation of datasets ===

image_datasets = {
    "train":    ImageFolderWithPaths(train_dir, transform=transform["train"]),
    "val":      ImageFolderWithPaths(val_dir, transform=transform["val"]),
    "test_id":  ImageFolderWithPaths(test_id_dir, transform=transform["test"]),
    "test_ood": ImageFolderWithPaths(test_ood_dir, transform=transform["test"]),
}

class_names = image_datasets["train"].classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

Classes: ['Apple___Apple_scab', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry___healthy', 'Corn_(maize)___healthy', 'Corn___Cercospora_leaf_spot', 'Corn___Common_rust', 'Corn___Northern_Leaf_Blight', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___healthy', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']
Number of classes: 30


In [8]:
# === Creation of data loaders ===

BATCH_SIZE = 256
NUM_WORKERS = 2 if is_colab() else 0

data_loaders = {
    "train": DataLoader(
        image_datasets["train"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS
    ),
    "val": DataLoader(
        image_datasets["val"],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    ),
    "test_id": DataLoader(
        image_datasets["test_id"],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    ),
    "test_ood": DataLoader(
        image_datasets["test_ood"],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    ),
}

In [9]:
# === Class weighting based on training set counts ===

train_targets = image_datasets["train"].targets
class_counts = np.bincount(train_targets)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

Class counts: [ 441  193 1151 1052  598  814  359  834  689  826  968  296  252  698
 1034  700  700  106  260 3563 1285  319 1489  700 1336  666 1240 3750
  261 1114]
Class weights: tensor([1.1323, 2.5873, 0.4338, 0.4747, 0.8350, 0.6134, 1.3909, 0.5987, 0.7247,
        0.6045, 0.5159, 1.6870, 1.9815, 0.7154, 0.4829, 0.7134, 0.7134, 4.7108,
        1.9206, 0.1401, 0.3886, 1.5654, 0.3354, 0.7134, 0.3738, 0.7498, 0.4027,
        0.1332, 1.9132, 0.4482], device='cuda:0')


In [10]:
# === Building model: ResNet18 from scratch ===

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [11]:
# =========================
# CONFIG
# =========================

load_existing_weights = False

nepochs = 40
patience = 6

CKPT_PATH = "./resnet18_scratch_best.pth"

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-2
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

In [12]:
# =========================
# TRAINING FUNCTION
# =========================

def train_model(model, data_loaders, optimizer, criterion, scheduler, nepochs, patience, save_path):
    history = {
        "train_losses": [],
        "val_losses": [],
        "train_accs": [],
        "val_accs": []
    }

    best_val_loss = float("inf")
    best_val_acc = 0.0
    epochs_no_improve = 0

    for epoch in range(nepochs):
        print(f"\nEpoch {epoch+1}/{nepochs}")
        print("-" * 40)

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0
            total = 0

            for images, labels, paths in data_loaders[phase]:
                images = images.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    preds = torch.argmax(outputs, dim=1)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * images.size(0)
                running_corrects += torch.sum(preds == labels).item()
                total += labels.size(0)

            epoch_loss = running_loss / total
            epoch_acc = running_corrects / total

            history[f"{phase}_losses"].append(epoch_loss)
            history[f"{phase}_accs"].append(epoch_acc)

            print(f"{phase:>5} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

            if phase == "val":
                scheduler.step(epoch_loss)

                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_val_acc = epoch_acc
                    epochs_no_improve = 0

                    torch.save({
                        "model": model.state_dict(),
                        "best_val_loss": best_val_loss,
                        "best_val_acc": best_val_acc,
                        "class_names": class_names
                    }, save_path)

                    print(f"Saved improved model to {save_path}")
                else:
                    epochs_no_improve += 1

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Current LR: {current_lr:.6g}")

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs.")
            break

    return history

In [ ]:
# =========================
# TRAIN OR LOAD
# =========================

if load_existing_weights and os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    print(f"Loaded existing checkpoint from {CKPT_PATH}")
    history = None
else:
    history = train_model(
        model=model,
        data_loaders=data_loaders,
        optimizer=optimizer,
        criterion=criterion,
        scheduler=scheduler,
        nepochs=nepochs,
        patience=patience,
        save_path=CKPT_PATH
    )


Epoch 1/40
----------------------------------------


In [ ]:
# === Training vs Validation Visualization ===

if history is not None:
    epochs = range(1, len(history["train_losses"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_losses"], label="Train Loss")
    plt.plot(epochs, history["val_losses"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Train vs Validation Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_accs"], label="Train Acc")
    plt.plot(epochs, history["val_accs"], label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Train vs Validation Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# === Load best checkpoint ===

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()

print(f"Loaded best checkpoint from {CKPT_PATH}")
print(f"Best validation loss: {ckpt.get('best_val_loss', 'N/A')}")
print(f"Best validation acc:  {ckpt.get('best_val_acc', 'N/A')}")

In [ ]:
# === Evaluation helper ===

def evaluate_model(model, loader, class_names, split_name="test"):
    model.eval()

    y_true = []
    y_pred = []

    correct_examples = []
    incorrect_examples = []

    with torch.no_grad():
        for images, labels, paths in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

            for i in range(images.size(0)):
                item = (
                    images[i].cpu(),
                    labels[i].cpu().item(),
                    preds[i].cpu().item(),
                    paths[i]
                )
                if preds[i] == labels[i]:
                    correct_examples.append(item)
                else:
                    incorrect_examples.append(item)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    overall_acc = (y_true == y_pred).mean()

    print(f"\n=== {split_name} Results ===")
    print(f"Overall accuracy: {overall_acc:.4f}")

    print("\nPer-class accuracy:")
    for i, class_name in enumerate(class_names):
        mask = (y_true == i)
        if mask.sum() == 0:
            acc = float("nan")
            print(f"{class_name:>20}: no samples")
        else:
            acc = (y_pred[mask] == y_true[mask]).mean()
            print(f"{class_name:>20}: {acc:.4f}  ({mask.sum()} samples)")

    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "overall_acc": overall_acc,
        "correct_examples": correct_examples,
        "incorrect_examples": incorrect_examples
    }

In [ ]:
# === Test on ID and OOD ===

id_results = evaluate_model(
    model,
    data_loaders["test_id"],
    class_names,
    split_name="Test ID"
)

ood_results = evaluate_model(
    model,
    data_loaders["test_ood"],
    class_names,
    split_name="Test OOD"
)

In [ ]:
# === Reverse normalization for visualization ===

def unnormalize(img):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return img * std + mean

In [ ]:
# === Plotting helper ===

def plot_examples(example_list, class_names, title, n=6):
    if len(example_list) == 0:
        print(f"No examples available for: {title}")
        return

    samples = random.sample(example_list, k=min(n, len(example_list)))

    plt.figure(figsize=(16, 4))

    for i, (img, t, p, path) in enumerate(samples, 1):
        ax = plt.subplot(1, len(samples), i)

        img_vis = unnormalize(img).permute(1, 2, 0).clamp(0, 1)
        ax.imshow(img_vis)
        ax.axis("off")

        fname = os.path.basename(path)
        ax.set_title(
            f"True: {class_names[t]}\nPred: {class_names[p]}\n{fname}",
            fontsize=9
        )

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# === Requested plots ===

plot_examples(
    id_results["correct_examples"],
    class_names,
    title="Correctly Classified Images - Test ID",
    n=6
)

plot_examples(
    id_results["incorrect_examples"],
    class_names,
    title="Incorrectly Classified Images - Test ID",
    n=6
)

plot_examples(
    ood_results["correct_examples"],
    class_names,
    title="Correctly Classified Images - Test OOD",
    n=6
)

plot_examples(
    ood_results["incorrect_examples"],
    class_names,
    title="Incorrectly Classified Images - Test OOD",
    n=6
)